# Biodiversity Hotspots

Processing the Biodiversity Hotspots dataset (version 2016.1). The dataset contains
36 polygon regions representing Earth's biologically richest and most endangered terrestrial
areas. Processing includes cleaning geometries and converting to mbtiles for upload to Mapbox.

**Source:** [Zenodo](https://doi.org/10.5281/zenodo.3261807)

**Citation:** Michael Hoffman, Kellee Koenig, Gill Bunting, Jennifer Costanza, & Williams, K. J. (2016). Biodiversity Hotspots (version 2016.1) (2016.1) [Data set]. Zenodo. https://doi.org/10.5281/zenodo.3261807

**License:** CC BY-SA 4.0

## Setup

In [ ]:
import json
import logging
import os
import subprocess
import time
from pathlib import Path

import antimeridian
import boto3
import geopandas as gpd
import requests
from dotenv import load_dotenv

load_dotenv()

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

path_raw = "../data/raw/hotspots_2016_1"
path_out = "../data/processed/biodiversity_hotspots"
os.makedirs(path_out, exist_ok=True)

In [ ]:
def create_mbtiles(
    source_path: Path,
    output_path: Path,
    layer_name: str,
    opts: str,
):
    """Use tippecanoe to create mbtiles from a GeoJSON file."""
    cmd = f"tippecanoe -o {output_path} -l {layer_name} {opts} {source_path}"
    logger.info(f"Running: {cmd}")
    r = subprocess.call(cmd, shell=True)
    if r != 0:
        raise RuntimeError(f"tippecanoe failed with exit code {r}")
    return r


def upload_mbtiles(mbtiles_path: str, tileset_id: str, name: str):
    """Upload an mbtiles file to Mapbox using the Uploads API."""
    token = os.environ["MAPBOX_ACCESS_TOKEN"]
    username = os.environ["MAPBOX_USERNAME"]
    base_url = f"https://api.mapbox.com/uploads/v1/{username}"
    full_tileset = f"{username}.{tileset_id}"

    file_size_mb = os.path.getsize(mbtiles_path) / (1024 * 1024)
    print(f"Uploading {mbtiles_path} ({file_size_mb:.1f} MB) → {full_tileset}")

    resp = requests.post(f"{base_url}/credentials", params={"access_token": token})
    resp.raise_for_status()
    creds = resp.json()

    s3 = boto3.client(
        "s3",
        aws_access_key_id=creds["accessKeyId"],
        aws_secret_access_key=creds["secretAccessKey"],
        aws_session_token=creds["sessionToken"],
        region_name="us-east-1",
    )
    s3.upload_file(mbtiles_path, creds["bucket"], creds["key"])

    resp = requests.post(
        base_url,
        params={"access_token": token},
        json={"url": creds["url"], "tileset": full_tileset, "name": name},
    )
    resp.raise_for_status()
    upload_id = resp.json()["id"]

    while True:
        time.sleep(5)
        resp = requests.get(f"{base_url}/{upload_id}", params={"access_token": token})
        resp.raise_for_status()
        status = resp.json()
        progress = status.get("progress", 0)
        print(f"  progress: {progress:.0%}", end="")
        if status.get("complete"):
            print(" — done!")
            return status
        if status.get("error"):
            print(f"\n  ERROR: {status['error']}")
            return status
        print()

## Load data

In [ ]:
hotspots = gpd.read_file(
    os.path.join(path_raw, "hotspots_2016_1.shp"), engine="pyogrio", on_invalid="warn"
)

print(f"Features: {len(hotspots)}")
print(f"CRS: {hotspots.crs}")
print(f"Columns: {list(hotspots.columns)}")
hotspots.head()

## Clean data

In [ ]:
if hotspots.crs and hotspots.crs.to_epsg() != 4326:
    hotspots = hotspots.to_crs(epsg=4326)

invalid_count = (~hotspots.geometry.is_valid).sum()
if invalid_count > 0:
    print(f"Fixing {invalid_count} invalid geometries")
    hotspots["geometry"] = hotspots.geometry.make_valid()

hotspots = hotspots[~hotspots.geometry.is_empty & hotspots.geometry.notna()]
hotspots = hotspots.drop_duplicates(subset="NAME")

print(f"Clean features: {len(hotspots)}")

## Export to GeoJSON and fix antimeridian

Polygons crossing the 180°/-180° line get rendered as spanning the entire globe in
GeoJSON. We fix those features by splitting them at the antimeridian.

In [ ]:
hotspots_geojson = os.path.join(path_out, "biodiversity_hotspots.geojson")

hotspots.to_file(hotspots_geojson, driver="GeoJSON")


def _crosses_antimeridian(feature):
    """Check if a feature's coordinates span the antimeridian."""
    def _extract_lons(coords):
        if isinstance(coords[0], (int, float)):
            return [coords[0]]
        lons = []
        for c in coords:
            lons.extend(_extract_lons(c))
        return lons

    lons = _extract_lons(feature["geometry"]["coordinates"])
    if not lons:
        return False
    return max(lons) - min(lons) > 180


with open(hotspots_geojson) as f:
    geojson = json.load(f)

fixed_count = 0
for i, feature in enumerate(geojson["features"]):
    if _crosses_antimeridian(feature):
        geojson["features"][i] = antimeridian.fix_geojson(feature)
        fixed_count += 1

with open(hotspots_geojson, "w") as f:
    json.dump(geojson, f)

print(f"Antimeridian: {fixed_count} features fixed, {len(geojson['features'])} total")

## Create mbtiles

Tippecanoe settings:
- `-Z1 -z10`: zoom levels 1 through 10
- `-B4`: base zoom 4 — all features guaranteed present at zoom 4+
- `--drop-densest-as-needed`: drop features at low zooms to stay under the tile size limit
- `--force`: overwrite existing mbtiles

In [ ]:
tippecanoe_opts = " ".join([
    "--force",
    "--read-parallel",
    "-Z1",
    "-z10",
    "-B4",
    "--drop-densest-as-needed",
    "--maximum-tile-bytes=1500000",
])

create_mbtiles(
    hotspots_geojson,
    os.path.join(path_out, "biodiversity_hotspots.mbtiles"),
    "biodiversity_hotspots",
    tippecanoe_opts,
)

## Upload to Mapbox

Requires `MAPBOX_ACCESS_TOKEN` (secret token with `uploads:read` and `uploads:write`
scopes) and `MAPBOX_USERNAME` in `../.env`.

Edit the `tileset_id` and `name` below before running.

In [ ]:
upload_mbtiles(
    mbtiles_path=os.path.join(path_out, "biodiversity_hotspots.mbtiles"),
    tileset_id="biodiversity_hotspots",  # ← edit this
    name="Biodiversity Hotspots",        # ← edit this
)